# 01 · Data audit

This notebook inspects the three NYPD extracts before any transformation. The key objective is to identify each table's **row grain** and avoid accidental duplication during joins.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
files = {
    "shootings": "shootings.csv",
    "victims": "shooting_victims.csv",
    "offenders": "shooting_offenders.csv",
}
tables = {name: pd.read_csv(RAW_DIR / filename) for name, filename in files.items()}

## Table grain and coverage

In [2]:
summary = pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in tables.items()},
        "unique_incidents": {
            name: frame["INCIDENT_KEY"].nunique() for name, frame in tables.items()
        },
        "duplicate_rows": {name: frame.duplicated().sum() for name, frame in tables.items()},
    }
)
summary.index.name = "table"
summary

,rows,unique_incidents,duplicate_rows
table,,,
shootings,23988,23988,0
victims,28748,23990,0
offenders,18750,14614,0


In [3]:
shootings = tables["shootings"]
incident_dates = pd.to_datetime(shootings["OCCUR_DATE"], format="%m/%d/%Y", errors="coerce")
pd.Series(
    {
        "first_incident": incident_dates.min().date(),
        "last_incident": incident_dates.max().date(),
        "invalid_or_missing_dates": incident_dates.isna().sum(),
    },
    name="coverage",
)

first_incident              2006-01-01
last_incident               2025-12-31
invalid_or_missing_dates             0
Name: coverage, dtype: object

## Missingness

In [4]:
missing = pd.concat(
    {
        name: frame.isna().sum().sort_values(ascending=False).head(6)
        for name, frame in tables.items()
    },
    names=["table", "column"],
).rename("missing_rows").to_frame()
missing["missing_pct"] = [
    100 * value / len(tables[table])
    for (table, _), value in missing["missing_rows"].items()
]
missing.round(2)

missing_rows  missing_pct
table     column                                       
shootings LOCATION_DESC              14344        59.80
          Longitude                    133         0.55
          Latitude                     133         0.55
          LOC_CLASSFCTN_DESC            32         0.13
          JURISDICTION_CODE              2         0.01
          LOC_OF_OCCUR_DESC              0         0.00
victims   VICTIM_ID                      1         0.00
          VICTIM_AGE_GROUP               1         0.00
          VICTIM_RACE                    1         0.00
          VICTIM_SEX                     1         0.00
          STAT_MURDER_FLG                1         0.00
          INCIDENT_KEY                   0         0.00
offenders PERP_AGE_GROUP                30         0.16
          INCIDENT_KEY                   0         0.00
          PERP_ID                        0         0.00
          PERP_SEX                       0         0.00
          PERP_RACE                      0         0.00

## Join-risk check

`shootings` is incident-level, while `victims` and `offenders` can each contain multiple rows for the same incident. Joining all three directly creates a many-to-many relationship and repeats victim outcomes. The analysis therefore:

- counts trends from the incident table;
- measures fatality at the victim level by joining victims to incidents (`many_to_one`);
- keeps offender analysis separate unless a question explicitly requires it.

In [5]:
naive_join = (
    tables["shootings"]
    .merge(tables["offenders"], on="INCIDENT_KEY", how="outer")
    .merge(tables["victims"], on="INCIDENT_KEY", how="outer")
)

pd.Series(
    {
        "victim_rows": len(tables["victims"]),
        "rows_after_naive_three_table_join": len(naive_join),
        "extra_rows_created": len(naive_join) - len(tables["victims"]),
    },
    name="join_check",
)

victim_rows                          28748
rows_after_naive_three_table_join    34148
extra_rows_created                    5400
Name: join_check, dtype: int64

## Audit conclusion

The source data supports incident-level trend analysis and victim-level fatality analysis, but not a single universal row grain. Missing offender attributes should be treated as **unknown**, not evidence that no offender existed. Demographic fields describe police administrative records and require cautious interpretation; they are not population rates.